# End of week 1 exercise

To demonstrate your familiarity with OpenAI API, and also Ollama, build a tool that takes a technical question,  
and responds with an explanation. This is a tool that you will be able to use yourself during the course!

In [11]:
# imports
import os
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import display, Markdown, update_display

load_dotenv(override=True)
api_key = os.getenv("OPENAI_API_KEY")
api_key_llama = os.getenv("LLAMA_API_KEY")




In [18]:
# constants
openai = OpenAI()
MODEL_GPT = 'gpt-4o-mini'
MODEL_LLAMA = 'llama3.2'

OLLAMA_BASE_URL = "http://localhost:11434/v1"
def model_wrapper(model):
  if model == 'gpt':
    return OpenAI(api_key=api_key)
  elif model == 'llama':
    return OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')
  else:
    raise ValueError(f"Invalid model: {model}")

system_prompt = """You are a helpful technical tutor with vast knowledge in software engineering, system design and architecture.
You are also a great teacher and can explain complex concepts in a way that is easy to understand.
You are also great at providing examples and analogies to help explain the concepts.
You are also great at providing code snippets and examples to help explain the concepts.
You are also great at providing tips and best practices to help explain the concepts.
"""
user_prompt =  """
Please explain what this concept in as easy to understand as possible:
"""




In [19]:
# set up environment
def get_user_prompt(question):
    return f"""
Please explain what this concept in as easy to understand as possible:
{question}
"""

def get_model(modelString):
  if modelString == 'gpt':
    return MODEL_GPT
  elif modelString == 'llama':
    return MODEL_LLAMA
  else:
    raise ValueError(f"Invalid model: {modelString}")

def get_response(question, modelString):
    message=[{"role":"system", "content": system_prompt}, {"role":"user", "content": get_user_prompt(question) }]
    model = get_model(modelString)
    stream= model_wrapper(modelString).chat.completions.create(
      model=model,
      messages=message,
      stream=True
    )
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)


    



In [20]:
# here is the question; type over this to ask something new

question = """
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)
"""

In [21]:
# Get gpt-4o-mini to answer, with streaming

get_response(question, 'gpt')

Let's break down the provided code step by step to make it easier to understand. This code appears to be a function that interacts with the OpenAI API to generate a brochure for a specific company and stream the results as they are generated.

### Function Overview
The `stream_brochure` function takes two arguments:
1. `company_name`: This is the name of the company for which the brochure is being generated.
2. `url`: This could be a link to the company's website or any relevant resource.

### How the Function Works
1. **Calling the OpenAI API**:
   - The line `stream = openai.chat.completions.create(...)` calls the OpenAI API to generate text based on the inputs provided. Here’s how it works:
     - **Model**: It specifies the AI model to use; in this case, it’s `gpt-4.1-mini`.
     - **Messages**: This is a list of messages that helps set context for the AI. There are two messages in the list:
       - The first message is from the **system**, which includes a system prompt (`brochure_system_prompt`) that likely defines what kind of text the model should generate (i.e., a brochure).
       - The second message is from the **user**, created by the function `get_brochure_user_prompt(company_name, url)`, which forms the user input for generating the brochure based on the specific company name and URL.
     - **stream=True**: This option indicates that the response will be streamed back chunk by chunk instead of being returned all at once. This is useful for long responses.

2. **Response Accumulation**:
   - The `response` variable is initialized as an empty string. This will accumulate the text generated by the model.
   
3. **Displaying the Response**:
   - `display_handle = display(Markdown(""), display_id=True)`: This line is used to prepare a display area for the generated text. The `Markdown("")` initializes the display with an empty Markdown element, and `display_id=True` allows the display to be updated later.
   
4. **Streaming the Response**:
   - The `for chunk in stream:` loop iterates over each chunk of text received from the OpenAI API.
     - `response += chunk.choices[0].delta.content or ''`: This appends the new content to the existing `response`. If there’s no content, it adds an empty string instead to avoid errors.
     - `update_display(Markdown(response), display_id=display_handle.display_id)`: This updates the displayed text with the accumulated `response` using Markdown formatting. The `display_id` is used to ensure the correct area of the UI is updated.

### Example Analogy
Think of the function as a live news ticker that updates in real-time as news reports come in. Here’s an analogy to understand it better:
- Imagine a journalist (the OpenAI API) is writing a company brochure. The editor (the function) gives the journalist the company name and website and asks for a brochure using a specific format (the system prompt).
- As the journalist writes, they send small updates to the editor instead of the whole brochure at once (streaming). The editor then updates the audience (the display) with these snippets as they come in, gradually revealing the complete brochure.

### Tips and Best Practices
- **Error Handling**: It’s a good idea to include error handling to manage situations when the API fails or returns unexpected data.
- **Chunk Processing**: If you expect very large responses, consider implementing a limit on how much text to accumulate before displaying to avoid overwhelming users.
- **UI/UX**: Make sure the UI refreshes smoothly to provide a good user experience as text is generated.

### Example Code Snippet
Here’s a very simplified version of how the function might look without the streaming and UI components (for clarity):

```python
def generate_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ]
    )
    return response.choices[0].message['content']
```

In this simplified version, we directly get the complete brochure response without streaming it, making it easier to understand how we can generate text in a straightforward manner. 

In summary, the original function is designed to create a brochure dynamically and continuously updates a display with the output as it's generated, enhancing the user experience significantly.

In [22]:
# Get Llama 3.2 to answer
get_response(question, 'llama')

**Explaining Generative Text Completion with OpenAI**

Let's break down what this Python function `stream_brochure` does:

**What it does:** The function generates a brochure description for a given company.

**Step-by-Step Explanation:**

1. **OpenAI API Call**: The function calls the OpenAI API, which is a powerful tool for generating text.
```python
stream = openai.chat.completions.create(
    ...
)
```
The `completions` endpoint allows us to specify a model (e.g., GPT-4) and input prompts. We'll discuss this in more detail later.

2. **Prepare Input Prompts**: The function prepares two input prompts:
	* `brochure_system_prompt`: A generic prompt that asks the AI to generate content.
	* `get_brochure_user_prompt(company_name, url)`: A custom prompt that includes the company name and URL. This is where we pass in our company's details.

```python
messages = [
    {"role": "system", "content": brochure_system_prompt},
    {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
]
```
The `messages` parameter allows us to specify multiple prompts to the API. We'll send both prompts in sequence.

3. **Streaming Output**: The function sets `stream=True`, which enables streaming mode for the OpenAI API. This means we can process the output chunk by chunk instead of waiting for the entire response.
```python
if True:
    ...  # create the completion API
```
When we call this method, it might take a while to get the full text. Instead, we'll retrieve it in fragments and update our display as we go.

4. **Assemble Response**: We start accumulating the output chunks into a single response string.
```python
response = ""
for chunk in stream:
    response += chunk.choices[0].delta.content or ''
```
We're ignoring any blank or invalid input chunks, but we'll keep including all the valid content.

5. **Update Display**: Finally, we update our display using `Markdown` and `display_id`.

```python
display_handle = display(Markdown(""), display_id=True)
update_display(Markdown(response), display_id=display_handle.display_id)
```
We're keeping track of where on the screen our output is.

**Code Snippet Example**

Here's a simple example for creating this `stream_brochure` function:

```python
import streamlit as st
from openai import chat

# Sample data
company_name = "John Doe Inc."
url = "https://www.johndoeinc.com"

def get_brochure_user_prompt(company_name, url):
    return f"{company_name} is a leading provider of innovative solutions. Their website: {url}"

def brochure_system_prompt():
    return "Please generate a brief company description..."

# Stream brochure functionality
stream_handle = st.empty_display()
with chat.openaim_chat(completions="gpt-4.1-mini", messages=[brochure_system_prompt, get_brochure_user_prompt(company_name, url)], stream=True) as out:
    for chunk in out:
        if isinstance(chunk, dict):
            response += chunk['choices'][0]['delta']['content']
            st.empty_display()
            st markdown(response)

# Keep track of the updated display
stream_handle.display_id
```
Feel free to try and adapt it according to your specific use case!